# {{PROJECT_NAME}}

KFP v2 eval-first fine-tuning pipeline on the Miramar platform.

**Development workflow:**
1. Edit `config.yaml` — set model ID, datasets, thresholds, and judge prompt
2. Edit `formatters.py` — add one formatter function per dataset, register in `FORMATTERS`
3. Write step logic in the `@dsl.component` cells below
4. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
5. Trigger **Deploy to KFP** workflow (or run **Compile & Submit** below)

**Pipeline DAG:**
```
prepare_dataset ──► baseline_eval ──► fine_tune ─┬─► post_finetune_eval ──►─┐
                                                 │                          │
                                                 └─► safety_eval ──────────►┤
                                                                            │
                                                                   deployment_gate
```

> `fine_tune` runs after `baseline_eval` (not parallel) — on single-node minikube, both steps
> need GPU memory simultaneously and will exceed the allocatable limit if run together.


## Step Development

Write step logic inside each `@dsl.component` function body.

- **All imports must be inside the function body** — each component runs in its own container
- Set `base_image` and `packages_to_install` to match the step's runtime requirements
- Use `Input[T]` / `Output[T]` to pass artifacts between steps
- GPU steps: call `.set_gpu_limit(1).set_memory_limit("64G")` in the pipeline cell

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

### prepare_dataset

Loads datasets from HuggingFace, applies formatters from `formatters.py`, splits into train/val/test.
The Build cell inlines `formatters.py` at the `# <<< FORMATTERS_INJECT >>>` marker.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "datasets<3.0",
        "huggingface_hub>=0.21.2,<0.24",
        "transformers",
    ],
)
def prepare_dataset(
    dataset_names: list,
    val_size: float,
    test_size: float,
    train_out: Output[Dataset],
    val_out: Output[Dataset],
    test_out: Output[Dataset],
):
    import json, pathlib, random

    # <<< FORMATTERS_INJECT >>>
    # formatters.py is inlined here by the Build cell. Do not remove this marker.

    # <<< LOADERS_INJECT >>>
    # loaders.py is inlined here by the Build cell. Do not remove this marker.

    missing = [n for n in dataset_names if n not in LOADERS]
    if missing:
        raise ValueError(f"No loader for: {missing}. Add to loaders.py LOADERS dict.")

    all_rows = []
    for name in dataset_names:
        ds = LOADERS[name]()
        all_rows.extend(
            {"instruction": r["instruction"], "response": r["response"], "source": r["source"]}
            for r in ds
        )

    random.shuffle(all_rows)
    n = len(all_rows)
    n_val  = max(1, int(n * val_size))
    n_test = max(1, int(n * test_size))
    train_rows = all_rows[n_val + n_test:]
    val_rows   = all_rows[:n_val]
    test_rows  = all_rows[n_val:n_val + n_test]

    pathlib.Path(train_out.path).write_text(json.dumps(train_rows))
    pathlib.Path(val_out.path).write_text(json.dumps(val_rows))
    pathlib.Path(test_out.path).write_text(json.dumps(test_rows))
    print(f"Dataset split: {len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test")

### baseline_eval

Evaluates the base model on the validation set before fine-tuning. Records metrics to MLflow.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "accelerate",
        "mlflow",
    ],
)
def baseline_eval(
    val: Input[Dataset],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
    profile: bool = False,
):
    import json, pathlib, sys, time, mlflow

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # IMPORTANT: always resolve a local snapshot path before calling from_pretrained().
    # Passing `base_model_id` (a Hub model ID string) triggers hf_hub_download, which
    # tries to create .locks/ on the HF cache PVC and fails with PermissionError
    # (pod runs as uid=0, treated as "other" on the 9p-mounted host filesystem).
    # A local path bypasses hf_hub_download entirely.
    def _local_model_path(model_id):
        cache = pathlib.Path("/root/.cache/huggingface/hub")
        key = model_id.replace("/", "--")
        commit = (cache / f"models--{key}" / "refs" / "main").read_text().strip()
        return str(cache / f"models--{key}" / "snapshots" / commit)

    model_path = _local_model_path(base_model_id)

    # ---- single source of truth for eval helpers ----
    _EVAL_HELPER_SRC = r"""
def extract_answer(text):
    pass  # TODO: implement — parse model output for the final answer letter/token

def _make_user_content(row):
    pass  # TODO: implement — wrap row["instruction"] in your prompt template
"""
    import pathlib as _pl
    _pl.Path("/tmp/eval_helpers.py").write_text(_EVAL_HELPER_SRC)
    _h = {}; exec(_EVAL_HELPER_SRC, _h)
    extract_answer = _h["extract_answer"]
    _make_user_content = _h["_make_user_content"]
    # ---- end eval helpers ----

    if profile:
        import subprocess as _sp
        json.dump({
            "val_data": val_data, "model_path": model_path,
            "mlflow_tracking_uri": mlflow_tracking_uri, "run_id": run_id,
            "max_new_tokens": max_new_tokens, "system_message": system_message,
            "do_sample": do_sample,
        }, open("/tmp/ns_state.json", "w"))

        out_dir = pathlib.Path(f"/nsight-reports/{{PROJECT_NAME}}/{run_id}/baseline-eval")
        _sp.run(["chmod", "-R", "777", "/nsight-reports"], check=False)
        out_dir.mkdir(mode=0o777, parents=True, exist_ok=True)
        out_dir.chmod(0o777)
        out_path = str(out_dir / "profile")

        worker = r"""
import json, torch, pathlib
import sys as _sys; _sys.path.insert(0, '/tmp')
from eval_helpers import extract_answer, _make_user_content
from transformers import AutoTokenizer, AutoModelForCausalLM
import mlflow as _mlflow

state = json.load(open("/tmp/ns_state.json"))
val_data = state["val_data"]
model_path = state["model_path"]
max_new_tokens = state["max_new_tokens"]
system_message = state["system_message"]
do_sample = state["do_sample"]

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
# NOTE: do NOT use load_in_4bit / BitsAndBytesConfig — bitsandbytes 4-bit is broken
# on Blackwell (sm_100, DGX Spark GB10); use dtype=torch.bfloat16 directly.
model.eval()

_mlflow.set_tracking_uri(state["mlflow_tracking_uri"])
_mlflow_run = _mlflow.start_run(run_name=f"{state['run_id']}-baseline-profile")
correct = 0
for i, row in enumerate(val_data):
    if i % 10 == 0:
        print(f"Profile progress: {i}/{len(val_data)}", flush=True)
        if i > 0:
            _mlflow.log_metric("running_accuracy", correct / i, step=i)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": _make_user_content(row)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
    if extract_answer(generated) == extract_answer(row["response"]): correct += 1

_mlflow.log_metric("running_accuracy", correct / len(val_data) if val_data else 0.0, step=len(val_data))
_mlflow.end_run()
json.dump({"correct": correct, "total": len(val_data)}, open("/tmp/ns_results.json", "w"))
print(f"Profile run done: {correct}/{len(val_data)}")
"""

        open("/tmp/ns_worker.py", "w").write(worker)
        _sp.run(["nsys", "profile", "--output", out_path,
                 "--force-overwrite=true", "--trace=cuda,nvtx,osrt",
                 sys.executable, "/tmp/ns_worker.py"], check=True)
        _res = json.loads(open("/tmp/ns_results.json").read())
        correct = _res["correct"]
        accuracy = correct / _res["total"] if _res["total"] else 0.0
        print(f"Profile report: {out_path}.nsys-rep")

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-baseline"):
            mlflow.log_param("eval_sample_size", len(val_data))
            mlflow.log_param("profiled", True)
            mlflow.log_metric("baseline_accuracy", accuracy)
    else:
        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-baseline"):
            mlflow.log_param("eval_sample_size", len(val_data))
            _t0 = time.perf_counter()

            # TODO: load base model
            # import torch
            # from transformers import AutoTokenizer, AutoModelForCausalLM
            # tokenizer = AutoTokenizer.from_pretrained(model_path)
            # model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16,
            #     device_map="auto", max_memory={0: "100GiB"})
            # model.eval()

            correct = 0
            for i, row in enumerate(val_data):
                if i % 10 == 0 and i > 0:
                    mlflow.log_metric("running_accuracy", correct / i, step=i)
                    mlflow.log_metric("seconds_per_sample", (time.perf_counter() - _t0) / i, step=i)
                messages = [
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": _make_user_content(row)},
                ]
                # TODO: run inference (uncomment after loading model above)
                # inputs = tokenizer.apply_chat_template(messages, return_tensors="pt",
                #     add_generation_prompt=True).to(model.device)
                # with torch.no_grad():
                #     out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
                # generated = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
                # if extract_answer(generated) == extract_answer(row["response"]): correct += 1

            accuracy = correct / len(val_data) if val_data else 0.0
            mlflow.log_metric("baseline_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_accuracy": accuracy}))
    print(f"Baseline accuracy: {accuracy:.4f}")

### fine_tune

Fine-tunes the base model using LoRA on the training set.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[],
)
def fine_tune(
    train: Input[Dataset],
    val: Input[Dataset],
    base_model_id: str,
    learning_rate: float,
    num_epochs: int,
    lora_r: int,
    lora_alpha: int,
    lora_dropout: float,
    target_modules: list,
    lora_bias: str,
    lora_task_type: str,
    batch_size: int,
    gradient_accumulation_steps: int,
    max_seq_length: int,
    bf16: bool,
    gradient_checkpointing: bool,
    eval_strategy: str,
    logging_steps: int,
    run_id: str,
    mlflow_tracking_uri: str,
    ft_model: Output[Model],
    profile: bool = False,
):
    import subprocess, sys, os
    # NGC containers set PIP_CONSTRAINT=/etc/pip/constraint.txt which pins pyarrow==19.0.1.
    # Clear it so pip can upgrade pyarrow to satisfy trl's datasets dependency.
    env = {**os.environ, "PIP_CONSTRAINT": ""}
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "pyarrow>=21.0.0", "transformers>=4.45,<5.0", "peft>=0.15.0", "trl>=0.14.0,<1.0", "accelerate>=0.27.0", "mlflow"],
        check=True, env=env,
    )
    import json, pathlib, mlflow

    train_data = json.loads(pathlib.Path(train.path).read_text())
    val_data   = json.loads(pathlib.Path(val.path).read_text())

    # IMPORTANT: always resolve a local snapshot path before calling from_pretrained().
    # Passing `base_model_id` triggers hf_hub_download → PermissionError on .locks/ PVC dir.
    def _local_model_path(model_id):
        cache = pathlib.Path("/root/.cache/huggingface/hub")
        key = model_id.replace("/", "--")
        commit = (cache / f"models--{key}" / "refs" / "main").read_text().strip()
        return str(cache / f"models--{key}" / "snapshots" / commit)

    model_path = _local_model_path(base_model_id)

    # TODO: implement to_chat — dataset-specific conversation formatter
    def to_chat(rows):
        pass  # returns HFDataset with "messages" column: [{role, content}, ...]

    if profile:
        import shutil
        import subprocess as _sp
        json.dump({
            "model_path": model_path,
            "train_data": train_data[:32],
            "val_data": val_data[:8],
            "learning_rate": learning_rate,
            "lora_r": lora_r, "lora_alpha": lora_alpha,
            "lora_dropout": lora_dropout, "target_modules": target_modules,
            "lora_bias": lora_bias, "lora_task_type": lora_task_type,
        }, open("/tmp/ns_state.json", "w"))

        out_dir = pathlib.Path(f"/nsight-reports/{{PROJECT_NAME}}/{run_id}/fine-tune")
        subprocess.run(["chmod", "-R", "777", "/nsight-reports"], check=False)
        out_dir.mkdir(mode=0o777, parents=True, exist_ok=True)
        out_dir.chmod(0o777)
        out_path = str(out_dir / "profile")

        worker = r"""
import json, torch, pathlib
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

state = json.load(open("/tmp/ns_state.json"))
model_path = state["model_path"]
train_data = state["train_data"]; val_data = state["val_data"]
learning_rate = state["learning_rate"]
lora_r = state["lora_r"]; lora_alpha = state["lora_alpha"]
lora_dropout = state["lora_dropout"]; target_modules = state["target_modules"]
lora_bias = state["lora_bias"]; lora_task_type = state["lora_task_type"]

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
lora_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    target_modules=target_modules, bias=lora_bias, task_type=lora_task_type)
model = get_peft_model(model, lora_config)

def to_chat(rows):
    pass  # TODO: project-specific chat formatter

train_ds = to_chat(train_data)
val_ds = to_chat(val_data)

# Profile worker overrides: minimal config (batch=1, seq=512, max_steps=3) for a brief trace.
training_args = SFTConfig(
    output_dir="/tmp/ft_profile_checkpoints",
    per_device_train_batch_size=1, gradient_accumulation_steps=1,
    learning_rate=learning_rate, bf16=True, gradient_checkpointing=True,
    max_steps=3, logging_steps=1, save_strategy="no", report_to="none",
    dataloader_num_workers=0, max_seq_length=512)
trainer = SFTTrainer(model=model, processing_class=tokenizer, args=training_args,
                     train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

pathlib.Path("/tmp/ft_profile_model").mkdir(parents=True, exist_ok=True)
model.save_pretrained("/tmp/ft_profile_model")
tokenizer.save_pretrained("/tmp/ft_profile_model")
print("Profile training complete (3 steps)")
"""

        open("/tmp/ns_worker.py", "w").write(worker)
        subprocess.run(["nsys", "profile", "--output", out_path,
                        "--force-overwrite=true", "--duration", "120",
                        "--trace=cuda,nvtx,osrt",
                        sys.executable, "/tmp/ns_worker.py"], check=True)
        print(f"Profile report: {out_path}.nsys-rep")
        pathlib.Path(ft_model.path).mkdir(parents=True, exist_ok=True)
        shutil.copytree("/tmp/ft_profile_model", ft_model.path, dirs_exist_ok=True)

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-finetune"):
            mlflow.log_params({"learning_rate": learning_rate, "num_epochs": num_epochs,
                               "lora_r": lora_r, "lora_alpha": lora_alpha, "profiled": True})
    else:
        # Working scaffold — fill in to_chat above, then this runs as-is
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from peft import LoraConfig, get_peft_model
        from trl import SFTTrainer, SFTConfig

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_path, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
        lora_cfg = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            target_modules=target_modules, bias=lora_bias, task_type=lora_task_type)
        model = get_peft_model(model, lora_cfg)
        training_args = SFTConfig(
            output_dir="/tmp/ft_checkpoints", num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            max_seq_length=max_seq_length, learning_rate=learning_rate,
            bf16=bf16, gradient_checkpointing=gradient_checkpointing,
            eval_strategy=eval_strategy, logging_steps=logging_steps,
            save_strategy="no", report_to="mlflow", dataloader_num_workers=0)
        trainer = SFTTrainer(model=model, processing_class=tokenizer, args=training_args,
            train_dataset=to_chat(train_data), eval_dataset=to_chat(val_data))

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-finetune"):
            mlflow.log_params({"learning_rate": learning_rate, "num_epochs": num_epochs,
                               "lora_r": lora_r, "lora_alpha": lora_alpha})
            trainer.train()
            history = trainer.state.log_history
            train_loss = next((e["loss"]      for e in reversed(history) if "loss"      in e), None)
            eval_loss  = next((e["eval_loss"] for e in reversed(history) if "eval_loss" in e), None)
            if train_loss is not None: mlflow.log_metric("train_loss", train_loss)
            if eval_loss  is not None: mlflow.log_metric("eval_loss",  eval_loss)

        pathlib.Path(ft_model.path).mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ft_model.path)
        tokenizer.save_pretrained(ft_model.path)
        print(f"Fine-tuning complete. Adapter saved to: {ft_model.path}")

### post_finetune_eval

Evaluates the fine-tuned model on the validation set. Results feed into the deployment gate.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "mlflow",
    ],
)
def post_finetune_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
    profile: bool = False,
):
    import json, pathlib, sys, time, mlflow

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # IMPORTANT: always resolve a local snapshot path before calling from_pretrained().
    # Passing `base_model_id` triggers hf_hub_download → PermissionError on .locks/ PVC dir.
    def _local_model_path(model_id):
        cache = pathlib.Path("/root/.cache/huggingface/hub")
        key = model_id.replace("/", "--")
        commit = (cache / f"models--{key}" / "refs" / "main").read_text().strip()
        return str(cache / f"models--{key}" / "snapshots" / commit)

    model_path = _local_model_path(base_model_id)

    # ---- single source of truth for eval helpers (same as baseline_eval) ----
    _EVAL_HELPER_SRC = r"""
def extract_answer(text):
    pass  # TODO: implement — same logic as baseline_eval

def _make_user_content(row):
    pass  # TODO: implement — same logic as baseline_eval
"""
    import pathlib as _pl
    _pl.Path("/tmp/eval_helpers.py").write_text(_EVAL_HELPER_SRC)
    _h = {}; exec(_EVAL_HELPER_SRC, _h)
    extract_answer = _h["extract_answer"]
    _make_user_content = _h["_make_user_content"]
    # ---- end eval helpers ----

    if profile:
        import subprocess as _sp
        json.dump({
            "val_data": val_data, "model_path": model_path, "ft_model_path": ft_model.path,
            "mlflow_tracking_uri": mlflow_tracking_uri, "run_id": run_id,
            "max_new_tokens": max_new_tokens, "system_message": system_message,
            "do_sample": do_sample,
        }, open("/tmp/ns_state.json", "w"))

        out_dir = pathlib.Path(f"/nsight-reports/{{PROJECT_NAME}}/{run_id}/post-finetune-eval")
        _sp.run(["chmod", "-R", "777", "/nsight-reports"], check=False)
        out_dir.mkdir(mode=0o777, parents=True, exist_ok=True)
        out_dir.chmod(0o777)
        out_path = str(out_dir / "profile")

        worker = r"""
import json, torch, pathlib
import sys as _sys; _sys.path.insert(0, '/tmp')
from eval_helpers import extract_answer, _make_user_content
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import mlflow as _mlflow

state = json.load(open("/tmp/ns_state.json"))
val_data = state["val_data"]; model_path = state["model_path"]
ft_model_path = state["ft_model_path"]
max_new_tokens = state["max_new_tokens"]; system_message = state["system_message"]
do_sample = state["do_sample"]

tokenizer = AutoTokenizer.from_pretrained(ft_model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
model = PeftModel.from_pretrained(model, ft_model_path)
model.eval()

_mlflow.set_tracking_uri(state["mlflow_tracking_uri"])
_mlflow_run = _mlflow.start_run(run_name=f"{state['run_id']}-postft-profile")
correct = 0
for i, row in enumerate(val_data):
    if i % 10 == 0:
        print(f"Profile progress: {i}/{len(val_data)}", flush=True)
        if i > 0:
            _mlflow.log_metric("running_accuracy", correct / i, step=i)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": _make_user_content(row)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
    if extract_answer(generated) == extract_answer(row["response"]): correct += 1

_mlflow.log_metric("running_accuracy", correct / len(val_data) if val_data else 0.0, step=len(val_data))
_mlflow.end_run()
json.dump({"correct": correct, "total": len(val_data)}, open("/tmp/ns_results.json", "w"))
print(f"Profile run done: {correct}/{len(val_data)}")
"""

        open("/tmp/ns_worker.py", "w").write(worker)
        _sp.run(["nsys", "profile", "--output", out_path,
                 "--force-overwrite=true", "--trace=cuda,nvtx,osrt",
                 sys.executable, "/tmp/ns_worker.py"], check=True)
        _res = json.loads(open("/tmp/ns_results.json").read())
        correct = _res["correct"]
        accuracy = correct / _res["total"] if _res["total"] else 0.0
        print(f"Profile report: {out_path}.nsys-rep")

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-postft-eval"):
            mlflow.log_param("eval_sample_size", len(val_data))
            mlflow.log_param("profiled", True)
            mlflow.log_metric("postft_accuracy", accuracy)
    else:
        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-postft-eval"):
            mlflow.log_param("eval_sample_size", len(val_data))
            _t0 = time.perf_counter()

            # TODO: load fine-tuned model (base + PeftModel adapter)
            # import torch
            # from transformers import AutoTokenizer, AutoModelForCausalLM
            # from peft import PeftModel
            # tokenizer = AutoTokenizer.from_pretrained(ft_model.path)
            # base = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16,
            #     device_map="auto", max_memory={0: "100GiB"})
            # model = PeftModel.from_pretrained(base, ft_model.path)
            # model.eval()

            correct = 0
            for i, row in enumerate(val_data):
                if i % 10 == 0 and i > 0:
                    mlflow.log_metric("running_accuracy", correct / i, step=i)
                    mlflow.log_metric("seconds_per_sample", (time.perf_counter() - _t0) / i, step=i)
                messages = [
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": _make_user_content(row)},
                ]
                # TODO: run inference
                # inputs = tokenizer.apply_chat_template(messages, return_tensors="pt",
                #     add_generation_prompt=True).to(model.device)
                # with torch.no_grad():
                #     out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
                # generated = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
                # if extract_answer(generated) == extract_answer(row["response"]): correct += 1

            accuracy = correct / len(val_data) if val_data else 0.0
            mlflow.log_metric("postft_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"postft_accuracy": accuracy}))
    print(f"Post-FT accuracy: {accuracy:.4f}")

### safety_eval

LLM-as-judge evaluation of the fine-tuned model's outputs. Uses `OPENAI_API_KEY` injected from the
`mlabs-api-keys` K8s secret. Judge model and system prompt are set in `config.yaml`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:25.03-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "openai",
        "mlflow",
    ],
)
def safety_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 256,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
    profile: bool = False,
):
    import json, pathlib, sys, mlflow
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # IMPORTANT: always resolve a local snapshot path before calling from_pretrained().
    # Passing `base_model_id` triggers hf_hub_download → PermissionError on .locks/ PVC dir.
    def _local_model_path(model_id):
        cache = pathlib.Path("/root/.cache/huggingface/hub")
        key = model_id.replace("/", "--")
        commit = (cache / f"models--{key}" / "refs" / "main").read_text().strip()
        return str(cache / f"models--{key}" / "snapshots" / commit)

    model_path = _local_model_path(base_model_id)

    if profile:
        import subprocess as _sp
        json.dump({
            "val_data": val_data, "model_path": model_path, "ft_model_path": ft_model.path,
            "max_new_tokens": max_new_tokens, "system_message": system_message,
            "do_sample": do_sample,
        }, open("/tmp/ns_state.json", "w"))

        out_dir = pathlib.Path(f"/nsight-reports/{{PROJECT_NAME}}/{run_id}/safety-eval")
        _sp.run(["chmod", "-R", "777", "/nsight-reports"], check=False)
        out_dir.mkdir(mode=0o777, parents=True, exist_ok=True)
        out_dir.chmod(0o777)
        out_path = str(out_dir / "profile")

        worker = r"""
import json, torch, pathlib
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

state = json.load(open("/tmp/ns_state.json"))
val_data = state["val_data"]; model_path = state["model_path"]
ft_model_path = state["ft_model_path"]
max_new_tokens = state["max_new_tokens"]; system_message = state["system_message"]
do_sample = state["do_sample"]

tokenizer = AutoTokenizer.from_pretrained(ft_model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
model = PeftModel.from_pretrained(model, ft_model_path)
model.eval()

responses = []
for i, row in enumerate(val_data):
    if i % 10 == 0:
        print(f"Profile progress: {i}/{len(val_data)}", flush=True)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": row["instruction"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated = tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
    responses.append({"instruction": row["instruction"], "generated": generated})

json.dump(responses, open("/tmp/ns_results.json", "w"))
print(f"Profile inference done: {len(responses)} responses")
"""

        open("/tmp/ns_worker.py", "w").write(worker)
        _sp.run(["nsys", "profile", "--output", out_path,
                 "--force-overwrite=true", "--trace=cuda,nvtx,osrt",
                 sys.executable, "/tmp/ns_worker.py"], check=True)
        print(f"Profile report: {out_path}.nsys-rep")

        _results = json.loads(open("/tmp/ns_results.json").read())
        client = OpenAI()
        # TODO: implement parse_score to extract a numeric score from judge JSON output
        scores = []
        for item in _results:
            result = client.chat.completions.create(
                model=judge_model_id,
                messages=[
                    {"role": "system", "content": judge_system_prompt},
                    {"role": "user",   "content": "Instruction: " + item["instruction"] + "\n\nResponse: " + item["generated"]},
                ],
                temperature=0.0,
            )
            # scores.append(parse_score(result.choices[0].message.content))
        avg_score = sum(scores) / len(scores) if scores else 0.0

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
            mlflow.log_metric("safety_avg_score", avg_score)
            mlflow.log_param("judge_model_id", judge_model_id)
            mlflow.log_param("profiled", True)
    else:
        # TODO: load fine-tuned model and generate responses, then score with judge LLM
        # import torch
        # from transformers import AutoTokenizer, AutoModelForCausalLM
        # from peft import PeftModel
        # tokenizer = AutoTokenizer.from_pretrained(ft_model.path)
        # base = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16,
        #     device_map="auto", max_memory={0: "100GiB"})
        # model = PeftModel.from_pretrained(base, ft_model.path)
        # client = OpenAI()  # uses OPENAI_API_KEY env var (injected from mlabs-api-keys)
        avg_score = 0.0  # placeholder

        mlflow.set_tracking_uri(mlflow_tracking_uri)
        with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
            mlflow.log_metric("safety_avg_score", avg_score)

    pathlib.Path(metrics.path).write_text(json.dumps({"safety_avg_score": avg_score}))
    print(f"Safety avg score: {avg_score:.4f}")

### deployment_gate

Compares baseline vs post-FT accuracy and safety score against thresholds from `config.yaml`.
Fails the pipeline if either check does not pass. On pass, pushes the adapter to GCS.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
)
def deployment_gate(
    test: Input[Dataset],
    ft_model: Input[Model],
    baseline_metrics: Input[Metrics],
    postft_metrics: Input[Metrics],
    safety_metrics: Input[Metrics],
    accuracy_delta_threshold: float,
    safety_score_threshold: float,
    gcs_bucket: str,
    run_id: str,
):
    import json, pathlib

    baseline = json.loads(pathlib.Path(baseline_metrics.path).read_text())
    postft   = json.loads(pathlib.Path(postft_metrics.path).read_text())
    safety   = json.loads(pathlib.Path(safety_metrics.path).read_text())

    # TODO: update metric keys to match what baseline_eval / post_finetune_eval log
    baseline_acc = baseline.get("baseline_accuracy", 0.0)
    postft_acc   = postft.get("postft_accuracy", 0.0)
    safety_score = safety.get("safety_avg_score", 0.0)

    delta = postft_acc - baseline_acc
    passed = delta >= -accuracy_delta_threshold and safety_score >= safety_score_threshold

    print(f"Accuracy delta : {delta:+.4f}  (threshold ≥ {-accuracy_delta_threshold:.4f})")
    print(f"Safety score   : {safety_score:.4f}  (threshold ≥ {safety_score_threshold:.4f})")
    print(f"Gate           : {'PASS' if passed else 'FAIL'}")

    if not passed:
        raise RuntimeError(
            f"Deployment gate failed — accuracy delta {delta:+.4f}, safety score {safety_score:.4f}"
        )

    # TODO: push ft_model adapter files to GCS
    # from google.cloud import storage
    # client = storage.Client()
    # bucket = client.bucket(gcs_bucket)
    # for p in pathlib.Path(ft_model.path).rglob("*"):
    #     bucket.blob(f"{run_id}/{p.relative_to(ft_model.path)}").upload_from_filename(str(p))
    print(f"Adapter would be pushed to gs://{gcs_bucket}/{run_id}/")

### Pipeline

Wire the steps together. Defaults are read from `config.yaml` at import time.
To override a default when submitting a run, pass it as an argument to `create_run_from_pipeline_package`
or set it in the KFP UI.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_dataset_names = [d["name"] for d in _cfg["datasets"]]

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline(
    base_model_id: str = _cfg["model"]["id"],
    judge_model_id: str = _cfg["judge"]["model"],
    judge_system_prompt: str = _cfg["judge"]["system_prompt"],
    dataset_names: list = _dataset_names,
    learning_rate: float = _cfg["training"]["learning_rate"],
    num_epochs: int = _cfg["training"]["num_epochs"],
    lora_r: int = _cfg["lora"]["r"],
    lora_alpha: int = _cfg["lora"]["alpha"],
    lora_dropout: float = _cfg["lora"]["dropout"],
    target_modules: list = _cfg["lora"]["target_modules"],
    lora_bias: str = _cfg["lora"]["bias"],
    lora_task_type: str = _cfg["lora"]["task_type"],
    batch_size: int = _cfg["training"]["batch_size"],
    gradient_accumulation_steps: int = _cfg["training"]["gradient_accumulation_steps"],
    max_seq_length: int = _cfg["training"]["max_seq_length"],
    bf16: bool = _cfg["training"]["bf16"],
    gradient_checkpointing: bool = _cfg["training"]["gradient_checkpointing"],
    eval_strategy: str = _cfg["training"]["eval_strategy"],
    logging_steps: int = _cfg["training"]["logging_steps"],
    val_size: float = _cfg["training"]["val_size"],
    test_size: float = _cfg["training"]["test_size"],
    eval_sample_size: int = _cfg["eval"]["sample_size"],
    safety_sample_size: int = _cfg["eval"]["safety_sample_size"],
    max_new_tokens: int = _cfg["eval"]["max_new_tokens"],
    safety_max_new_tokens: int = _cfg["eval"]["safety_max_new_tokens"],
    do_sample: bool = _cfg["eval"]["do_sample"],
    system_message: str = _cfg["eval"]["system_message"],
    accuracy_delta_threshold: float = _cfg["eval"]["accuracy_delta_threshold"],
    safety_score_threshold: float = _cfg["eval"]["safety_score_threshold"],
    gcs_bucket: str = _cfg["deployment"]["gcs_bucket"],
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    profile_baseline: bool = False,
    profile_finetune: bool = False,
    profile_postft: bool = False,
    profile_safety: bool = False,
):
    prep = prepare_dataset(
        dataset_names=dataset_names,
        val_size=val_size,
        test_size=test_size,
    )

    base_eval = baseline_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
        profile=profile_baseline,
    )
    base_eval.set_gpu_limit(1).set_memory_limit("64G")

    ft = fine_tune(
        train=prep.outputs["train_out"],
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        lora_bias=lora_bias,
        lora_task_type=lora_task_type,
        batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        max_seq_length=max_seq_length,
        bf16=bf16,
        gradient_checkpointing=gradient_checkpointing,
        eval_strategy=eval_strategy,
        logging_steps=logging_steps,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        profile=profile_finetune,
    )
    # fine_tune runs after baseline_eval — on single-node minikube, both steps need GPU
    # memory simultaneously and will exceed the allocatable limit if scheduled in parallel.
    ft.after(base_eval)
    ft.set_gpu_limit(1).set_memory_limit("96G")

    post_eval = post_finetune_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
        profile=profile_postft,
    )
    post_eval.set_gpu_limit(1).set_memory_limit("64G")

    safety = safety_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        max_new_tokens=safety_max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
        profile=profile_safety,
    )
    safety.set_gpu_limit(1).set_memory_limit("64G")

    gate = deployment_gate(
        test=prep.outputs["test_out"],
        ft_model=ft.outputs["ft_model"],
        baseline_metrics=base_eval.outputs["metrics"],
        postft_metrics=post_eval.outputs["metrics"],
        safety_metrics=safety.outputs["metrics"],
        accuracy_delta_threshold=accuracy_delta_threshold,
        safety_score_threshold=safety_score_threshold,
        gcs_bucket=gcs_bucket,
        run_id=run_id,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = [
        "OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY",
        "WANDB_API_KEY", "LANGCHAIN_API_KEY", "NGC_API_KEY", "NVIDIA_API_KEY",
    ]
    _key_map = {k: k for k in _SECRET_KEYS}
    _HF_CACHE_PVC = "hf-model-cache"
    _NSIGHT_PVC = "nsight-reports"
    for _task in [base_eval, ft, post_eval, safety]:
        k8s_ext.mount_pvc(_task, pvc_name=_HF_CACHE_PVC, mount_path="/root/.cache/huggingface")
        k8s_ext.mount_pvc(_task, pvc_name=_NSIGHT_PVC, mount_path="/nsight-reports")

    for _task in [prep, base_eval, ft, post_eval, safety, gate]:
        k8s_ext.use_secret_as_env(_task, _SECRET, _key_map)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

What this does:
1. Reads `formatters.py` and inlines it into the `prepare_dataset` component body
2. Copies all other step cells unchanged
3. Copies the pipeline cell (which reads `config.yaml` at import time)
4. Writes `pipeline.py`

In [ ]:
import json, pathlib, re, textwrap


def build_pipeline(
    notebook_path="notebook.ipynb",
    formatters_path="formatters.py",
):
    nb = json.loads(pathlib.Path(notebook_path).read_text())
    formatters_src = pathlib.Path(formatters_path).read_text().rstrip("\n")

    step_srcs, pipeline_src = [], None
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = cell.get("metadata", {}).get("tags", [])
        src = "".join(cell["source"])
        if "kfp_step" in tags:
            if "# <<< FORMATTERS_INJECT >>>" in src:
                indented = textwrap.indent(formatters_src, "    ")
                src = src.replace("    # <<< FORMATTERS_INJECT >>>", indented)
            step_srcs.append(src)
        elif "kfp_pipeline" in tags:
            pipeline_src = src

    if not step_srcs:
        raise RuntimeError("No cells tagged 'kfp_step' found.")
    if pipeline_src is None:
        raise RuntimeError("No cell tagged 'kfp_pipeline' found.")

    names = [
        m.group(1)
        for src in step_srcs
        for m in [re.search(r"^def (\w+)\(", src, re.MULTILINE)]
        if m
    ]

    out  = "# Generated by notebook.ipynb \u2014 do not edit manually.\n"
    out += "# Re-run the Build cell to regenerate.\n\n"
    out += "from kfp import dsl\n"
    out += "from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact\n\n\n"
    out += "\n\n\n".join(step_srcs)
    out += "\n\n\n"
    out += pipeline_src
    out += "\n"
    pathlib.Path("pipeline.py").write_text(out)
    print(f"Wrote pipeline.py \u2014 {len(names)} component(s): {', '.join(names)}")


build_pipeline()

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)